## Imports


In [30]:
import json
import os
from langdetect import detect, LangDetectException
from sentence_transformers import SentenceTransformer
import faiss
from pathlib import Path
from sklearn.metrics import precision_score, recall_score, f1_score

## Datasets

In [13]:
DATA_DIR = Path().resolve().parent

### Query-Response Mapper

In [ ]:
MAPPER_PATH = DATA_DIR / "Datasets" / "mapper" / "mapper.json"

In [25]:
MAPPER_PATH

WindowsPath('E:/Studies/MIT/7/FYP/Project/Datasets/mapper/mapper.json')

In [23]:
with open(MAPPER_PATH, "r", encoding="utf-8") as f:
    MAPPER = json.load(f)

### Testing Dataset

In [26]:
TESTER_PATH = DATA_DIR / "Datasets" / "mapper" / "test.json"

In [31]:
with open(TESTER_PATH, "r", encoding="utf-8") as f:
    TESTER = json.load(f)

## Embedder

In [10]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

## CaneSpeak

In [34]:
def cane_speak(question_text, disease_name="Healthy", score_threshold=0.50):
    qtext = question_text.strip()

    try:
        lang = detect(qtext)
    except LangDetectException:
        lang = "en"

    candidates = []
    for entry in MAPPER:
        disease = entry.get("disease", "").lower()
        if disease == disease_name.lower() or disease == "healthy":
            for qa in entry.get("quesans", []):
                subs = qa.get("sub_questions", {})
                
                if isinstance(subs, dict):
                    for lang_key, sub_list in subs.items():
                        for sq in sub_list:
                            candidates.append({
                                "canonical": qa.get("canonical_question", ""),
                                "sub_question": sq,
                                "answer": qa.get("answer", {}),
                                "lang_key": lang_key,
                            })
                else:
                    for sq in subs:
                        candidates.append({
                            "canonical": qa.get("canonical_question", ""),
                            "sub_question": sq,
                            "answer": qa.get("answer", {}),
                            "lang_key": "en",
                        })

    if not candidates:
        return {"success": False, "error": "No candidates found."}

    corpus = []
    for c in candidates:
        corpus.append(c["canonical"])
        corpus.append(c["sub_question"])

    corpus_emb = embedder.encode(corpus, convert_to_numpy=True)
    d = corpus_emb.shape[1]
    index = faiss.IndexFlatIP(d)
    faiss.normalize_L2(corpus_emb)
    index.add(corpus_emb)

    q_emb = embedder.encode([qtext], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)

    scores, idx = index.search(q_emb, k=1)
    best_vec_index = int(idx[0][0])
    best_candidate_index = best_vec_index // 2
    best_score = float(scores[0][0])

    result = candidates[best_candidate_index]

    if best_score < score_threshold:
        return {
            "success": False,
            "detected_language": lang,
            "score": best_score,
            "message": f"No sufficiently confident match. (Score={best_score:.2f})"
        }

    answer_field = result["answer"]
    if isinstance(answer_field, str):
        answer_text = answer_field
    else:
        answer_text = answer_field.get("en") or list(answer_field.values())[0]

    return {
        "success": True,
        "detected_language": lang,
        "score": best_score,
        "canonical_question": result["canonical"],
        "matched_sub_question": result["sub_question"],
        "answer_text": answer_text,
        "disease_name_used": disease_name
    }

## Test Function

In [36]:
def test_canespeak(tests = TESTER):

    correct = 0
    total = len(tests)

    y_true = []
    y_pred = []

    for item in tests:
        q = item["question"]
        expected_disease = item["disease"].lower()

        res = cane_speak(q, disease_name=item["disease"])
        # print("\nQ:", q)

        if res["success"]:
            # print("Matched:", res["canonical_question"])
            # print("Answer :", res["answer_text"])
            # print("Score  :", res["score"])

            canonical = res["canonical_question"].lower()
            answer = res["answer_text"].lower()

            disease_key = expected_disease[:3]

            match = disease_key in canonical or disease_key in answer

            y_true.append(1)
            y_pred.append(1 if match else 0)

            if match:
                correct += 1
                # print("Correct.")
            else:
                pass
                # print("Incorrect mapping.")
        else:
            pass
            # print("No confident match. Score:", res.get("score"))
            y_true.append(1)
            y_pred.append(0)

    accuracy = correct / total if total > 0 else 0
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"Accuracy:  {accuracy:.2%}")
    print(f"Precision: {precision:.2%}")
    print(f"Recall:    {recall:.2%}")
    print(f"F1 Score:  {f1:.2%}")


## Testing and Results

In [37]:
accuracy = test_canespeak()
accuracy

Accuracy:  87.78%
Precision: 100.00%
Recall:    87.78%
F1 Score:  93.49%
